In [ ]:
import psycopg2
from sqlalchemy import create_engine

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from db_config import CONN_STR

engine = create_engine(CONN_STR)

query = """
SELECT
    m.movie_id,
    m.title,
    m.year,
    m.genre,
    m.rating,
    m.runtime_minutes,
    m.budget,
    m.country,
    mp.average_rating,
    mp.number_of_votes,
    mp.approval_index,
    mp.production_budget,
    mp.worldwide_gross,
    mp.domestic_gross,
    bo.worldwide,
    bo.domestic,
    bo."foreign",
    bo.year AS box_office_year,
    COUNT(DISTINCT oa.fact_id) FILTER (WHERE oa.winner IS TRUE) AS oscar_wins,
    COUNT(DISTINCT oa.fact_id) AS oscar_nominations,
    COUNT(DISTINCT ms.star_id) AS star_count,
    COUNT(DISTINCT mw.writer_id) AS writer_count
FROM main.dim_movie m
LEFT JOIN main.fact_movie_performance mp ON m.movie_id = mp.movie_id
LEFT JOIN main.fact_box_office bo ON m.movie_id = bo.movie_id
LEFT JOIN main.fact_oscar_awards oa ON m.movie_id = oa.movie_id
LEFT JOIN main.movie_stars ms ON m.movie_id = ms.movie_id
LEFT JOIN main.movie_writers mw ON m.movie_id = mw.movie_id
GROUP BY
    m.movie_id, m.title, m.year, m.genre, m.rating, m.runtime_minutes, m.budget,
    m.country, mp.average_rating, mp.number_of_votes, mp.approval_index,
    mp.production_budget, mp.worldwide_gross, mp.domestic_gross,
    bo.worldwide, bo.domestic, bo."foreign", bo.year;
"""


df = pd.read_sql(query, engine)

print(df.shape)
print(df.head()
      
      )



In [ ]:
key_columns = [
    'budget', 'production_budget', 'worldwide_gross',
    'average_rating', 'number_of_votes', 'approval_index'
]
df_filtered = df.dropna(subset=key_columns, how='all')  # прибираємо, якщо ВСІ ключові - NaN

print(f"Залишилося рядків: {df_filtered.shape[0]}")



In [ ]:

# Підготовка — обираємо лише колонки з пропусками
missing_cols = df_filtered.columns[df_filtered.isnull().any()]

plt.figure(figsize=(12, 6))
sns.heatmap(df_filtered[missing_cols].isnull(), cbar=False, cmap='viridis')
plt.title('Heatmap пропущених значень у збережених фільмах')
plt.show()


In [ ]:
numeric_cols = [
    'budget', 'production_budget',
    'worldwide_gross', 'domestic_gross',
    'worldwide', 'domestic', 'foreign'
]

plt.figure(figsize=(14, 8))
df_filtered[numeric_cols].boxplot()
plt.xticks(rotation=45)
plt.title('Boxplot основних числових показників (збори, бюджет)')
plt.ylabel('Сума у $')
plt.yscale('log')  # логарифмічна шкала — бо дуже великі значення
plt.grid(True)
plt.show()


In [ ]:
print(df_filtered[numeric_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))


In [ ]:

# Гістограма розподілу жанрів
plt.figure(figsize=(12, 6))
df['genre'].value_counts().sort_values(ascending=False).plot(kind='bar')
plt.title('Розподіл фільмів за жанрами')
plt.xlabel('Жанр')
plt.ylabel('Кількість фільмів')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Гістограма бюджету
plt.figure(figsize=(10, 5))
sns.histplot(df['budget'].dropna(), bins=50, kde=True)
plt.title('Розподіл бюджету фільмів')
plt.xlabel('Бюджет (USD)')
plt.ylabel('Кількість фільмів')
plt.show()
